# Train Whisper từ đầu (Train from scratch) cho Tiếng Việt

Sử dụng dataset `google/fleurs` tiếng Việt. Đã cập nhật để khắc phục các lỗi thư viện mới.

In [ ]:
!pip install --upgrade pip
# BẮT BUỘC dùng datasets < 3.0.0 để tránh lỗi 'Dataset scripts are no longer supported'
!pip install "datasets<3.0.0" transformers accelerate evaluate jiwer librosa soundfile

## 1. Kết nối Google Drive để lưu model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/STT_Vietnamese_Model_Scratch'

## 2. Chuẩn bị dữ liệu

In [ ]:
from datasets import load_dataset, Audio

print('Đang tải dữ liệu...')
train_dataset = load_dataset('google/fleurs', 'vi_vn', split='train[:5%]', trust_remote_code=True)
test_dataset = load_dataset('google/fleurs', 'vi_vn', split='test[:5%]', trust_remote_code=True)

train_dataset = train_dataset.cast_column('audio', Audio(sampling_rate=16000))
test_dataset = test_dataset.cast_column('audio', Audio(sampling_rate=16000))

## 3. Khởi tạo Processor (từ Whisper chuẩn)

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained('openai/whisper-small')
tokenizer = WhisperTokenizer.from_pretrained('openai/whisper-small', language='vi', task='transcribe')
processor = WhisperProcessor.from_pretrained('openai/whisper-small', language='vi', task='transcribe')

## 4. Xử lý dữ liệu

In [ ]:
def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = feature_extractor(audio['array'], sampling_rate=audio['sampling_rate']).input_features[0]
    text = batch.get('transcription', batch.get('raw_transcription', batch.get('sentence', '')))
    batch['labels'] = tokenizer(text).input_ids
    return batch

train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
test_dataset = test_dataset.map(prepare_dataset, remove_columns=test_dataset.column_names)

## 5. Khởi tạo Mô hình MỚI (Từ đầu)

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import WhisperConfig, WhisperForConditionalGeneration

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': feature['input_features']} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        label_features = [{'input_ids': feature['labels']} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')

        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

config = WhisperConfig.from_pretrained('openai/whisper-small')
model = WhisperForConditionalGeneration(config)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

## 6. Metrics và Huấn luyện

In [ ]:
import evaluate
metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer}

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4, 
    warmup_steps=500,
    max_steps=1000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=200,
    eval_steps=200,
    logging_steps=25,
    report_to=['tensorboard'],
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print('Bắt đầu huấn luyện từ đầu...')
trainer.train()

model.save_pretrained(OUTPUT_DIR + '/final')
processor.save_pretrained(OUTPUT_DIR + '/final')
print('Hoàn thành! Model đã được lưu tại Drive.')